# AI-Based Constrained Payment Routing Optimization

### Final showcase notebook

This notebook is the **main submission artifact**. It does not merely demonstrate API calls: it connects to the real project data/checkpoints in Google Drive, reconstructs the key analyses, loads the frozen experiment outputs, and walks through an actual online routing decision in detail.

**Design principle:** each layer of complexity was added because a simpler formulation failed.

`static prediction → temporal memory → continuous memory → stage-specific prediction → LP oracle → online pressure policy → prospective A/B validation`

## 0. Setup — public GitHub repo + Google Drive

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil

REPO_URL = "https://github.com/orankedem/flexfactor-routing-final-project.git"
LOCAL_REPO = Path("/content/flexfactor-routing-final-project")

if LOCAL_REPO.exists() and not (LOCAL_REPO/".git").exists():
    shutil.rmtree(LOCAL_REPO)
if not LOCAL_REPO.exists():
    subprocess.run(["git","clone",REPO_URL,str(LOCAL_REPO)],check=True)
else:
    subprocess.run(["git","-C",str(LOCAL_REPO),"pull"],check=True)

os.chdir(LOCAL_REPO)
sys.path.insert(0,str(LOCAL_REPO)) if str(LOCAL_REPO) not in sys.path else None
print("Repository ready:",LOCAL_REPO)

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

DRIVE_ROOT = Path("/content/gdrive/MyDrive")
print("Drive mounted:", DRIVE_ROOT.exists())

## 1. Locate the real project artifacts

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.showcase import (
    discover_artifacts, load_candidate_universe,
    detailed_pressure_trace
)

ART = discover_artifacts(DRIVE_ROOT)
display(pd.DataFrame({"artifact":ART.keys(),"path":[str(v) if v else "NOT FOUND" for v in ART.values()]}))

assert ART["data"] is not None, "Standardized data not found in Drive"

# Part I — The financial decision problem

FlexFactor attempts to recover declined card transactions through alternative payment routes. A route is a processor/provider/sponsor-bank combination.

For transaction $i$, candidate route $r$, and time $t$:

$$
\hat p_{ir,t}=P(\text{success}\mid X_i,r,H_t,\text{prior attempt state}).
$$

The goal is not merely accurate classification. The system must **choose a route online** while respecting finite route capacity.

## 2. Load the actual standardized attempt-level data

In [ ]:
attempts=pd.read_parquet(ART["data"])
attempts["timestamp"]=pd.to_datetime(attempts["timestamp"],utc=True)
print(f"Attempt rows: {len(attempts):,}")
print(f"Logical transactions: {attempts['transaction_id'].nunique():,}" if 'transaction_id' in attempts else "")

stage_summary=(attempts.groupby('attempt')
               .agg(rows=('success','size'), success_rate=('success','mean'))
               .reset_index())
display(stage_summary)

# Part II — Why routing is non-static

Two questions must be answered before building the optimizer:

1. Does route choice contain useful conditional signal?
2. Does the environment drift over time?

The second question changes the architecture fundamentally: if success behavior changes, a static merchant/issuer/route representation is insufficient.

## 3. Concept drift — observed approval rate through time

In [ ]:
a1=attempts[attempts['attempt'].eq(1)].copy()
a1['month']=a1['timestamp'].dt.strftime('%Y-%m')
monthly=a1.groupby('month').agg(rows=('success','size'),success_rate=('success','mean')).reset_index()
display(monthly)
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(monthly['month'],monthly['success_rate'],marker='o')
ax.set(title='A1 approval rate changes over time',xlabel='Month',ylabel='Observed success rate')
ax.tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.show()

## 4. Route signal after basic traffic-mix adjustment

In [ ]:
# Descriptive/predictive diagnostic — NOT a causal estimate.
if 'amount' in a1.columns:
    amount_col='amount'
elif 'amount_numeric' in a1.columns:
    amount_col='amount_numeric'
else:
    amount_col=None
merchant_col=next((c for c in ['merchant_id','Order_MerchantId','merchant'] if c in a1.columns),None)
route_col=next((c for c in ['route','route_clean','PaymentProvider_GroupId'] if c in a1.columns),None)

if amount_col and merchant_col and route_col:
    tmp=a1[[merchant_col,route_col,'month','success',amount_col]].copy()
    tmp['amount_decile']=pd.qcut(pd.to_numeric(tmp[amount_col],errors='coerce'),10,duplicates='drop')
    keys=[merchant_col,'month','amount_decile']
    tmp['group_baseline']=tmp.groupby(keys,observed=True)['success'].transform('mean')
    tmp['residual']=tmp['success']-tmp['group_baseline']
    route_res=(tmp.groupby(route_col)
               .agg(rows=('success','size'),raw_success=('success','mean'),residual=('residual','mean'))
               .query('rows >= 1000')
               .sort_values('residual',ascending=False))
    display(route_res.head(12))
else:
    print('Canonical merchant/route/amount columns differ; use the already validated project residual analysis.')

# Part III — Model selection: why XGBoost?

The project compared model families on the **same frozen A1 architecture and chronological development folds** before tuning the winner.

- CatBoost: native categorical handling
- LightGBM: efficient gradient boosting baseline
- XGBoost: final selected family
- linear SGD/logistic model: sanity baseline

The notebook loads the original comparison table from Drive when available, rather than inventing a comparison after the fact.

## 5. Original model-family comparison

In [ ]:
model_sel=ART['a1_model_selection']
arch_path=(model_sel/'summary/architecture_aggregate.csv') if model_sel else None
if arch_path and arch_path.exists():
    arch=pd.read_csv(arch_path)
    display(arch)
else:
    print('Original architecture_aggregate.csv not found. Historical progression: LightGBM → CatBoost experiments → final XGBoost.')

# Part IV — Giving a tabular model memory

A static model sees $X_t$ but not what has been happening recently. The first solution was fixed rolling windows. Their weakness is an arbitrary cutoff: an observation just inside the window receives full weight, while one just outside disappears.

Instead, historical relevance fades continuously:

$$w(\Delta t)=0.5^{\Delta t/h}.$$

Multiple half-lives $h\in\{3,14,60,180\}$ expose several speeds of change to XGBoost.

## 6. Continuous-memory intuition

In [ ]:
ages=np.arange(0,181)
fig,ax=plt.subplots(figsize=(8,4))
for h in [3,14,60,180]:
    ax.plot(ages,0.5**(ages/h),label=f'{h}d half-life')
ax.set(title='Continuous temporal memory',xlabel='Age of historical observation (days)',ylabel='Weight')
ax.legend(); plt.tight_layout(); plt.show()

### Leakage rule

For a transaction at time $t$, historical state is emitted **before** adding outcomes at $t$. Thus all temporal features obey the information set available at decision time. A2 may additionally use the observed A1 response; A3 may use A1 and A2 state.

# Part V — Final A1/A2/A3 predictive models

## 7. Factual chronological predictive performance

In [ ]:
predictive=pd.read_csv('artifacts/reference_predictive_metrics.csv')
display(predictive.style.format({
    'success_rate':'{:.2%}','roc_auc':'{:.3f}','average_precision':'{:.3f}',
    'logloss_skill':'{:.1%}','brier':'{:.4f}','ece':'{:.4f}'
}))

These are **factual observed-route metrics**: outcomes actually occurred. They validate ranking and probability quality separately from any counterfactual routing claim.

# Part VI — What did the models actually use?

## 8. Feature-family importance across A1/A2/A3

In [ ]:
# Prefer the exact saved feature-importance artifact. Fall back to the frozen
# headline table copied from the completed experiment.
fi_roots=[p for p in [ART['feature_policy'], DRIVE_ROOT/'flexfactor_policy_v2/feature_importance_lp_online_v1'] if p]
fi_path=None
for r in fi_roots:
    for rel in ['feature_importance/feature_family_importance.csv','feature_importance/feature_family_importance.csv']:
        p=Path(r)/rel
        if p.exists(): fi_path=p; break
    if fi_path: break

if fi_path:
    fi=pd.read_csv(fi_path)
    display(fi)
else:
    fi=pd.read_csv('artifacts/reference_feature_family_importance.csv')
    display(fi)

# Clean visual using the frozen family summary.
ref_fi=pd.read_csv('artifacts/reference_feature_family_importance.csv')
for stage in ['A1','A2','A3']:
    g=ref_fi[ref_fi.stage.eq(stage)].sort_values('normalized_gain_pct')
    fig,ax=plt.subplots(figsize=(7,3.6))
    ax.barh(g['family'],g['normalized_gain_pct'])
    ax.set(title=f'{stage} — XGBoost importance by feature family',xlabel='Normalized total gain (%)')
    plt.tight_layout(); plt.show()

## 9. Top individual features from the frozen models

In [ ]:
top_path=None
for r in fi_roots:
    p=Path(r)/'feature_importance/feature_importance_all.csv'
    if p.exists(): top_path=p; break
if top_path:
    raw_fi=pd.read_csv(top_path)
    for attempt in [1,2,3]:
        g=raw_fi[raw_fi['attempt'].eq(attempt)].sort_values('normalized_total_gain',ascending=False).head(15)
        print(f'A{attempt} top features')
        display(g[['feature','family','normalized_total_gain']])
else:
    print('Exact individual-feature CSV not found in Drive; family-level frozen importance is shown above.')

# Part VII — Prediction is not the decision

If capacity were unlimited, choose the route with the highest $\hat p$. But route volumes may move only around ±30% from historical volume. This couples decisions across transactions.

The offline LP solves the entire period jointly:

$$\max_x\sum_{i,r}x_{ir}\hat p_{ir}$$

or, for approved transaction value:

$$\max_x\sum_{i,r}x_{ir}Amount_i\hat p_{ir}.$$

## 10. LP oracle — absolute meaning of the 100% benchmark

In [ ]:
policy=pd.read_csv('artifacts/reference_policy_results.csv')
display(policy)
print('100% LP success opportunity = +188.777 model-implied approvals relative to historical routing.')
print('Success-optimal LP also adds ≈1.299M expected approved transaction value.')
print('Value-optimal LP adds ≈1.424M expected approved transaction value.')

**Important:** “100%” means the maximum model-implied benefit found by the full-hindsight LP under the frozen probability model and capacity constraints. It is not 100% transaction success and it is not causal proof.

# Part VIII — Online routing: greedy vs capacity-aware pressure

Greedy ignores future scarcity. The pressure policy gives an over-used route a dynamic shadow price:

$$Pressure_{r,t}=\frac{A_{r,t}-B_{r,t}}{\max(0.3B_{r,t},1)}$$

$$Score_{ir}=\hat p_{ir}-\lambda Pressure_{r,t}.$$

The adjusted score chooses the route; evaluation still uses the **raw calibrated model probability**.

## 11. Opportunity capture and capacity behavior

In [ ]:
show=policy[policy.policy.isin(['greedy','success_pressure','LP_success'])].copy()
fig,ax=plt.subplots(figsize=(7.5,4))
ax.bar(show['policy'],100*show['success_capture'])
ax.set(title='Share of model-implied LP success opportunity captured',ylabel='Opportunity captured (%)',ylim=(0,105))
plt.tight_layout(); plt.show()

capacity=pd.DataFrame({
    'policy':['Greedy','Success pressure λ=0.10','Balanced pressure λ=0.15','Value pressure λ=0.20'],
    'routes_near_30pct_boundary':[9,1,1,1],
    'mean_abs_route_deviation':[.2636,.1026,.0760,.0753]
})
display(capacity)

# Part IX — Detailed real inference walkthrough

This section uses the **actual saved candidate predictions from the June chronological backtest**. It automatically finds an event where the highest raw-probability route is *not* the pressure-policy choice, then reconstructs the decision from the cumulative route state immediately before that event.

This is the clearest demonstration of the difference between **prediction** and **decision optimization**.

## 12. Load actual candidate scores and explain one decision

In [ ]:
assert ART['backtest'] is not None, 'Backtest candidate checkpoints not found in Drive'
candidates=load_candidate_universe(ART['backtest'],month='2026-06')
print(f'Candidate rows loaded: {len(candidates):,}')
meta,decision_table=detailed_pressure_trace(candidates,lambda_=0.10)
display(pd.Series(meta,name='decision'))
display(decision_table.style.format({
    'raw_p_success':'{:.4f}','pressure':'{:+.3f}','adjusted_score':'{:.4f}'
}))

Read the table left to right:

1. **raw_p_success** — what the frozen XGBoost model believes about each candidate route;
2. **baseline / policy cumulative counts** — current route usage state;
3. **pressure** — scarcity/over-use signal;
4. **adjusted_score** — probability minus shadow price;
5. **feasible** — whether the strict development capacity guard allows the choice;
6. **selected** — route chosen online.

The important point is that the model does not directly “choose” the route. It provides calibrated probabilities to a separate constrained decision layer.

# Part X — Robustness rather than one magic λ

## 13. Later-period robustness table

In [ ]:
rob=ART['robustness']
val_path=rob/'summary/validation_overall.csv' if rob else None
if val_path and val_path.exists():
    validation=pd.read_csv(val_path)
    display(validation)
    daily_path=rob/'summary/validation_daily_rollup.csv'
    if daily_path.exists(): display(pd.read_csv(daily_path))
else:
    print('Frozen validation headline: on Jun 8–14 the main pressure policies beat greedy on both modeled approvals and approved value on all 7/7 days.')

# Part XI — Scientific interpretation

Following discussion with the data scientist, offline policy evaluation is centered on the model's calibrated probability estimates. That makes the comparison coherent and realistic **under the frozen model**.

But only $Y_i(R_{logged})$ is observed. $Y_i(R_{alternative})$ is counterfactual. Therefore:

- AUC/AP/log-loss/Brier/ECE → **factual predictive evidence**;
- capacity/movement → **operational evidence**;
- LP/pressure uplift → **model-implied opportunity**;
- true causal production lift → requires **prospective controlled A/B testing**.

## 14. Final validation-status table

In [ ]:
display(pd.DataFrame([
['Predictive quality','Chronologically validated on observed routes'],
['Calibration','Measured on factual route outcomes'],
['Capacity behavior','Validated in chronological replay'],
['LP / pressure gains','Counterfactual, calibrated-model-implied'],
['True production uplift','Requires controlled online/A-B test'],
],columns=['layer','status']))

# Conclusion

This is not merely an XGBoost classifier. It is a dynamic financial decision system:

**concept drift → temporal state → stage-specific probability estimation → constrained optimization → online scarcity pricing → controlled live validation.**

The main online result is that moderate capacity pressure captures roughly **85–86% of the model-implied LP opportunity**, while behaving much more sensibly with route capacity than naive greedy routing.